# DIVRS — Run (ML-10M)

Code đã sửa sẵn trong project, data ML-10M đã có ở `data/ml10m/output/`. Notebook chỉ **cài lib + chạy**.

- Bật GPU: **Runtime → Change runtime type → T4 GPU**.
- Notebook phải chạy với thư mục làm việc = gốc project (chỗ chứa `app.py`).

## 1. GPU + lib

In [1]:
!nvidia-smi -L
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

GPU 0: Tesla T4 (UUID: GPU-e71503cf-307e-e538-14ee-1501b73f80c1)
torch 2.11.0+cu128 | CUDA: True


## 2. Kiểm tra data đã sẵn

In [2]:
import os
D = 'data/ml10m/output/'
need = ['train_coo_record.npz','val_coo_record.npz','test_coo_record.npz',
        'train_skew_coo_record.npz','popularity.npy','popularity_blend.npy']
miss = [f for f in need if not os.path.exists(D+f)]
print('cwd =', os.getcwd())
print('Data thieu:', miss if miss else 'KHONG — du, san sang!')

cwd = /content
Data thieu: ['train_coo_record.npz', 'val_coo_record.npz', 'test_coo_record.npz', 'train_skew_coo_record.npz', 'popularity.npy', 'popularity_blend.npy']


## 3. Train + Test — ML-10M / DIVRS (MF)
`epochs=30` để chạy thử nhanh; bỏ flag đó để chạy đầy đủ (mặc định 500 + early-stop).

In [3]:
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --output ./output/ \
  --use_gpu=True --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=30 --es_patience=3

python3: can't open file '/content/app.py': [Errno 2] No such file or directory


## 4. Ghi chú
- **Baseline so sánh**: đổi `--flagfile` → `config/ml10m_mf.cfg` / `ml10m_ips.cfg` / `ml10m_cause.cfg` / `ml10m_lgn.cfg`.
- **GCN** (`ml10m_DIVRS-GCN.cfg`): cần cài `dgl` khớp torch; file `*_coo_adj_graph.npz` đã có sẵn trong data.
- **Douban**: chưa có data (phải tự build qua pipeline dps) — để sau.
- Kết quả Recall@K/NDCG in ra log; lưu ở `./output/`.